In [0]:
# =============================================================
# nb_audit_queries.py
# Layer   : Audit
# Purpose : Demo-friendly queries for validation output.
#
# Run after nb_run_fact_validation.py.
# =============================================================

from pyspark.sql import functions as F


def _widget(name: str, default: str) -> str:
    try:
        value = dbutils.widgets.get(name).strip()
        if value:
            return value
    except Exception:
        pass
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


RUN_ID = _widget("run_id", "")
TABLE_NAME = _widget("table_name", "ALL_FACTS")
LOAD_DATE = _widget("load_date", "")
STORAGE_ACCOUNT = _widget("storage_account", "petiot")
AUDIT_CONTAINER = _widget("audit_container", "audit")


def _run_notebook_with_fallback(label: str, paths: list[str]) -> None:
    last_error = None
    for path in paths:
        try:
            print(f"[loader] Loading {label}: {path}")
            get_ipython().run_line_magic("run", path)
            print(f"[loader] Loaded {label}: {path}")
            return
        except Exception as exc:
            last_error = exc
            print(f"[loader] Could not load {path}: {str(exc)[:250]}")
    raise RuntimeError(f"Unable to load {label}. Last error: {last_error}")


_run_notebook_with_fallback(
    "set_auth_context",
    [
        "/Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb",
        "/Workspace/Shared/pet-analytics/shared/set_auth_context",
        "/Shared/pet-analytics/shared/set_auth_context",
    ],
)

_run_notebook_with_fallback(
    "audit_config",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_config",
        "/Workspace/Shared/pet-analytics/audit/audit_config.py",
        "/Shared/pet-analytics/audit/audit_config",
    ],
)

_run_notebook_with_fallback(
    "audit_writer",
    [
        "/Workspace/Shared/pet-analytics/audit/audit_writer",
        "/Workspace/Shared/pet-analytics/audit/audit_writer.py",
        "/Shared/pet-analytics/audit/audit_writer",
    ],
)
ensure_audit_container_access.__globals__.update({'audit_path': audit_path, 'dbutils': dbutils, 'spark': spark})


ensure_audit_container_access(STORAGE_ACCOUNT, AUDIT_CONTAINER)

run_summary = read_audit_table("run_summary", STORAGE_ACCOUNT, AUDIT_CONTAINER)
table_summary = read_audit_table("table_validation_summary", STORAGE_ACCOUNT, AUDIT_CONTAINER)
rule_results = read_audit_table("rule_results", STORAGE_ACCOUNT, AUDIT_CONTAINER)

if RUN_ID:
    run_summary = run_summary.filter(F.col("run_id") == RUN_ID)
    table_summary = table_summary.filter(F.col("run_id") == RUN_ID)
    rule_results = rule_results.filter(F.col("run_id") == RUN_ID)

if LOAD_DATE:
    table_summary = table_summary.filter(F.col("load_date") == normalize_date(LOAD_DATE))
    rule_results = rule_results.filter(F.col("load_date") == normalize_date(LOAD_DATE))

if TABLE_NAME and TABLE_NAME.upper() not in {"ALL", "ALL_FACTS", "*"}:
    table_summary = table_summary.filter(F.col("table_name") == TABLE_NAME)
    rule_results = rule_results.filter(F.col("table_name") == TABLE_NAME)

print("=" * 80)
print("Query 1: Latest audit runs")
print("=" * 80)
display(
    run_summary
      .orderBy(F.col("_audit_write_ts").desc())
      .select(
          "run_id", "status", "load_date", "table_filter",
          "tables_requested", "tables_failed", "failed_rules",
          "warning_rules", "duration_seconds", "_audit_write_ts"
      )
)

print("=" * 80)
print("Query 2: Table validation summary")
print("=" * 80)
display(
    table_summary
      .orderBy("table_name")
      .select(
          "run_id", "table_name", "load_date", "dq_status",
          "bronze_rows_for_load_date", "silver_rows_for_business_date",
          "bronze_minus_silver_business_date", "row_drop_pct",
          "failed_rules", "warning_rules", "observed_bronze_anomaly_rules"
      )
)

print("=" * 80)
print("Query 3: Failed and warning rules")
print("=" * 80)
display(
    rule_results
      .filter(F.col("status").isin("FAIL", "ERROR", "WARN"))
      .orderBy("table_name", "severity", "rule_id")
      .select(
          "table_name", "rule_id", "rule_name", "layer", "severity",
          "status", "measured_count", "total_count", "measured_pct",
          "recommendation"
      )
)

print("=" * 80)
print("Query 4: Bronze anomalies observed but expected to be cleaned")
print("=" * 80)
display(
    rule_results
      .filter(F.col("status") == "OBSERVED")
      .orderBy(F.col("measured_count").desc())
      .select(
          "table_name", "rule_id", "rule_name", "measured_count",
          "measured_pct", "description", "recommendation"
      )
)

print("=" * 80)
print("Query 5: Validation scorecard by table")
print("=" * 80)
display(
    rule_results
      .groupBy("table_name", "status")
      .count()
      .groupBy("table_name")
      .pivot("status")
      .sum("count")
      .na.fill(0)
      .orderBy("table_name")
)

print("=" * 80)
print("Query 6: Full rule-level audit trail")
print("=" * 80)
display(
    rule_results
      .orderBy("table_name", "rule_id")
      .select(
          "run_id", "table_name", "load_date", "rule_id", "rule_name",
          "layer", "severity", "status", "measured_count", "total_count",
          "measured_pct", "description", "recommendation"
      )
)